In [ ]:
from pycanal import Canal
import pandas as pd 
import os 
from Bio import SeqIO
from Bio import AlignIO
from scipy import stats
import seaborn as sns
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
hMP_AF_orthologdb = pd.read_pickle('hMP_AF_orthologdb.pkl')
hTM_ortologs = pd.read_pickle('hTM_ortologs.pkl')
dSAMs = pd.read_pickle('../files/dSAM_MPs_v_paper_v2_new_locs.pkl')
AF2 = pd.read_pickle('../files/AF2_with_ENSG.pkl')

In [ ]:
for i in dSAMs.index:
    dSAMs.loc[i,'start'] = AF2[AF2.acc==dSAMs.loc[i,'acc']].AAseq.str.find(dSAMs.loc[i,'AAseq']).values[0]
    dSAMs.loc[i,'end'] = AF2[AF2.acc==dSAMs.loc[i,'acc']].AAseq.str.find(dSAMs.loc[i,'AAseq']).values[0] +len(dSAMs.loc[i,'AAseq'])

In [ ]:
acc_number= pd.DataFrame()
for i in tqdm(set(hTM_ortologs.human)):
    acc_number=acc_number.append({'acc':i,'val':len(hTM_ortologs[hTM_ortologs.human==i])},ignore_index=True)
    

In [ ]:
def para2dist(acc,tree_cutoff=.7):#MSA 2 tree
    file = '/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/hTM_AF2_panther_orthologs/MSAs/%s/%s_msa.fa'%(acc,acc)
    if os.path.isfile(file)==False:
        return pd.DataFrame()
    evo_dist = pd.DataFrame()
    fasta_sequences = SeqIO.parse(open(file),'fasta')
    print(fasta_sequences)
    for fasta in fasta_sequences:
        name, sequence = fasta.id, str(fasta.seq)
        print(name)
        #tax  = hMP_AF_orthologdb[hMP_AF_orthologdb.acc==name]['acc'].values[0].split('_')[1]
        tax  = hMP_AF_orthologdb[hMP_AF_orthologdb.acc==name]['acc'].values[0].split('_')[1]
        evo_dist = evo_dist.append({'acc':name,'tax':tax,'AAseq':sequence},ignore_index=True)
    calculator = DistanceCalculator('identity')
    msa = AlignIO.read(file, 'fasta')
    #dm = calculator.get_distance(msa)
    #evo_dist['tree'] = dm[0]
    #evo_dist = evo_dist.sort_values(by='tree')
    return msa#evo_dist[evo_dist.tree<=tree_cutoff]

In [ ]:
msa

In [ ]:
def msa2cons_scores(acc):
    file = '/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/hTM_AF2_panther_orthologs/MSAs/%s/%s_msa.fa'%(acc,acc)
    if  os.path.isfile(file)==False:
        return pd.DataFrame()
    else:
        canal = Canal(fastafile=file, #Multiple sequence alignment (MSA) of homologous sequences
                      ref=0, #Position of reference sequence in MSA, use first sequence
                      startcount=1, #Position label of first residue in reference sequence
                      verbose=False # Print out progress
                      )
        cons_scores = canal.analysis(include=None, method='all')
        xx = []
        x = pd.read_pickle('/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/webserver_sandbox/human_AF2/%s.pkl'%acc)
        for i in x.index:
            if (x.loc[i,'AF2_SS_helix_count']==0) and (x.loc[i,'AF2_pLDDT_max']<70) and(x.loc[i,'AF2_accessibility']=='***********'):
                xx.append({'pos':int(x.loc[i,'position']),'ele':'IDR'})
            if (x.loc[i,'AF2_pLDDT_max']>70)&(x.loc[i,'AF2_accessibility']=='***********'):
                xx.append({'pos':int(x.loc[i,'position']),'ele':'Structred_exposed'})
            if (x.loc[i,'AF2_pLDDT_max']>70)&(x.loc[i,'AF2_accessibility']=='...........'):
                xx.append({'pos':int(x.loc[i,'position']),'ele':'Structred_burid'})
        x3 = pd.DataFrame(xx)
        for i in cons_scores.index:
            cons_scores.loc[i,'pos']=i
            if x3[x3.pos==i].empty == False:
                cons_scores.loc[i,'ele'] = x3[x3.pos==i].ele.values[0]
        for dSAM in dSAMs[dSAMs.acc=='Q9NQW8'].index:
            cons_scores.loc[cons_scores[cons_scores.pos.isin(range(int(dSAMs.loc[dSAM,'start']),int(dSAMs.loc[dSAM,'end'])+1))].index,'ele']='dSAM'
        cons_scores['acc']=acc
        return cons_scores 

In [ ]:
df

In [ ]:
df = pd.DataFrame()
for acc in tqdm(set(dSAMs.acc)):
    if (acc=='Q6UXD1') or (acc=='A8MZ97'):
        continue
    else:
        df = pd.concat([df,msa2cons_scores(acc)],ignore_index=True)

In [ ]:
#df.to_pickle('concervation_msa.pkl')
df = pd.read_pickle('concervation_msa.pkl')

In [ ]:
sns.pointplot(x='ele',y='relative',data=df,order=['IDR','Structred_exposed','dSAM','Structred_burid',])
#plt.ylim([1,3.5])

In [ ]:
l = []
for acc in tqdm(set(df.acc)):
    for ele in ['IDR', 'Structred_burid', 'Structred_exposed', 'dSAM']:
        l.append({'acc':acc,'val':df[(df.ele==ele)&(df.acc==acc)].relative.mean(),'ele':ele})
        
#sns.stripplot(x='ele',y='val',data=pd.DataFrame(l),order=['IDR','Structred_exposed','dSAM','Structred_burid'])


In [ ]:
df

In [ ]:
fig, ax = plt.subplots(figsize=(3, 1.5))
ax = sns.pointplot(x='ele',y='val',data=pd.DataFrame(l),order=['IDR','Structred_exposed','dSAM','Structred_burid'],color='gray')
ax.set_xticklabels(['$Disorder$','$Structured^{exposed}$','mdSAM','$Structured^{buried}$'],rotation=45)
plt.ylabel('$Conservation$')#Kullback-Leibler\;divergence
ax.grid(color='#B4B5B4')

In [ ]:
xx = pd.DataFrame(l).pivot_table(index='acc',columns='ele',values='val')

In [ ]:
stats.ttest_rel(xx.dropna()['IDR'],xx.dropna()['dSAM'])

In [ ]:
len(set(pd.DataFrame(l).acc))

In [ ]:
df[df.]

In [ ]:
sns.stripplot(x='ele',y='relative',data=df.dropna(),alpha=0.01)#[df.acc=='Q9UI33']

In [ ]:
l = []
for acc in tqdm(set(df.acc)):
    for ele in ['IDR', 'Structred_burid', 'Structred_exposed', 'dSAM']:
        l.append({'acc':acc,'val':df[(df.ele==ele)&(df.acc==acc)].relative.mean(),'ele':ele})
        
#sns.stripplot(x='ele',y='val',data=pd.DataFrame(l),order=['IDR','Structred_exposed','dSAM','Structred_burid'])
fig, ax = plt.subplots(figsize=(3, 3))
ax = sns.pointplot(x='ele',y='val',data=pd.DataFrame(l),order=['IDR','Structred_exposed','dSAM','Structred_burid'],color='gray')
ax.set_xticklabels(['$Disorder$','$Structured^{exposed}$','mdSAM','$Structured^{buried}$'],rotation=45)
plt.ylabel('$H(shannon)$')
ax.grid(color='#B4B5B4')

In [ ]:
e = pd.DataFrame(l)
fig, ax = plt.subplots(figsize=(2, 2))
#ax.grid(color='#B4B5B4')
sns.set(font_scale=.8,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})

ax = sns.stripplot(y='ele',x='val',data=e,
                 order=['IDR','Structred_exposed','dSAM','Structred_burid'],palette=['#E8B658','#477cbd','#C03830','#477cbd'],alpha=.03)
ax = sns.boxplot(y='ele',x='val',data=e,
                 order=['IDR','Structred_exposed','dSAM','Structred_burid'],color = "w",showfliers = False,whis=.0,notch=True, showcaps=False)
ax.set_yticklabels(['$Disorder$','$Structured^{exposed}$','$mdSAM$','$Structured^{buried}$'],rotation=0)#=e[e.acc.isin(acc_number[(acc_number.val>=1)&(acc_number.val<=1000)].acc)]
plt.xlabel('$Conservation$')#Kullback-Leibler\;divergence
ax.locator_params(axis="x", nbins=4)
ax.locator_params(axis="y", nbins=4)  
plt.xlim([1.5,3])

#plt.ylabel('$Residue\;type$')
#plt.savefig('Conservation.svg',bbox_inches='tight',transparent=True)

# fig, ax = plt.subplots(figsize=(2, 1))
# sns.stripplot(acc_number[acc_number.acc.isin(list(set(dSAMs.acc)))],orient='h',alpha=0.1,color='grey')
# plt.title('$MSA\;depth$')
# plt.xscale('log')

In [ ]:
ll = []
for acc in tqdm(set(df.acc)):
    for ele in ['IDR', 'Structred_burid', 'Structred_exposed', 'dSAM']:
        ll.append({'acc':acc,'conservation':df[(df.ele==ele)&(df.acc==acc)].relative.mean(),'ele':ele,'Conservation_type':'Kullback-Leibler'})
        ll.append({'acc':acc,'conservation':1/df[(df.ele==ele)&(df.acc==acc)].shannon.mean(),'ele':ele,'Conservation_type':'shannon'})
        ll.append({'acc':acc,'conservation':df[(df.ele==ele)&(df.acc==acc)].lockless.mean(),'ele':ele,'Conservation_type':'lockless'})
conservation_df = pd.DataFrame(ll)
        #sns.stripplot(x='ele',y='val',data=pd.DataFrame(l),order=['IDR','Structred_exposed','dSAM','Structred_burid'])
# fig, ax = plt.subplots(figsize=(3, 3))
# ax = sns.pointplot(x='ele',y='val',data=pd.DataFrame(l),order=['IDR','Structred_exposed','dSAM','Structred_burid'],color='gray')
# ax.set_xticklabels(['$Disorder$','$Structured^{exposed}$','mdSAM','$Structured^{buried}$'],rotation=45)
# plt.ylabel('$H(shannon)$')
# ax.grid(color='#B4B5B4')

In [ ]:
fig = plt.figure(figsize = (5,2))#20,10
axs = fig.subplots(nrows=1, ncols=3,gridspec_kw = {'hspace':.1,'wspace':.10})
sns.boxplot(y='ele',x='conservation',data=conservation_df[conservation_df.Conservation_type=='Kullback-Leibler'],
                 order=['IDR','Structred_exposed','dSAM','Structred_burid'],ax=axs[0],color='#B4B5B4',showfliers = False,whis=.5,notch=True, showcaps=False)
sns.boxplot(y='ele',x='conservation',data=conservation_df[conservation_df.Conservation_type=='shannon'],
                 order=['IDR','Structred_exposed','dSAM','Structred_burid'],ax=axs[1],color='#B4B5B4',showfliers = False,whis=.5,notch=True, showcaps=False)
sns.boxplot(y='ele',x='conservation',data=conservation_df[conservation_df.Conservation_type=='lockless'],
                 order=['IDR','Structred_exposed','dSAM','Structred_burid'],color='#B4B5B4',ax=axs[2],showfliers = False,whis=.5,notch=True, showcaps=False)
axs[0].set_title('$Kullback-Leibler$',fontsize=8)
axs[1].set_title('$Shannon$',fontsize=8)
axs[2].set_title('$Lockless$',fontsize=8)
axs[0].legend([],frameon=False)
axs[1].legend([],frameon=False)
axs[2].legend([],frameon=False)
axs[2].set_yticklabels([])
axs[1].set_yticklabels([])
axs[0].set_yticklabels(['$Disorder$','$Structured^{exposed}$','$mdSAM$','$Structured^{buried}$'])
axs[2].set_ylabel('')
axs[1].set_ylabel('')
axs[0].set_ylabel('$Residue\;type$')
axs[0].grid(color='#B4B5B4')
axs[1].grid(color='#B4B5B4')
axs[2].grid(color='#B4B5B4')
axs[1].set_xlabel('$\\frac{1}{H}$')
axs[0].set_xlabel('$Divergence$')
axs[2].set_xlabel('$Conservation$')
axs[0].text(1.5,3.35,'$n=929$',fontsize=8)
plt.savefig('Conservation_new_version.svg',bbox_inches='tight',transparent=True)


In [ ]:
fig = plt.figure(figsize = (4,2))#20,10
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.10})
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})

# sns.boxplot(y='ele',x='conservation',data=conservation_df[conservation_df.Conservation_type=='Kullback-Leibler'],
#                  order=['IDR','Structred_exposed','dSAM','Structred_burid'],ax=axs[0],color='#B4B5B4',showfliers = False,whis=.5,notch=True, showcaps=False)
sns.boxplot(y='ele',x='conservation',data=conservation_df[conservation_df.Conservation_type=='shannon'],
                 order=['IDR','Structred_exposed','dSAM','Structred_burid'],ax=axs[0],color='#B4B5B4',showfliers = False,whis=.5,notch=True, showcaps=False)
sns.boxplot(y='ele',x='conservation',data=conservation_df[conservation_df.Conservation_type=='lockless'],
                 order=['IDR','Structred_exposed','dSAM','Structred_burid'],color='#B4B5B4',ax=axs[1],showfliers = False,whis=.5,notch=True, showcaps=False)
axs[0].set_title('$Shannon$',fontsize=16)
axs[1].set_title('$Lockless$',fontsize=16)
axs[0].legend([],frameon=False)
axs[1].legend([],frameon=False)
axs[1].set_yticklabels([])
axs[0].set_yticklabels(['$Disorder$','$Structured^{exposed}$','$mdSAM$','$Structured^{buried}$'])
axs[1].set_ylabel('')
axs[0].set_ylabel('$Residue\;type$')
axs[0].grid(color='#B4B5B4')
axs[1].grid(color='#B4B5B4')
axs[0].set_xlabel('$\\frac{1}{H}$')
axs[1].set_xlabel('$Conservation$')
axs[0].text(3.5,-.05,'$n=929$',fontsize=12)
plt.savefig('Conservation_new_version_short.svg',bbox_inches='tight',transparent=True)


In [ ]:
print(stats.ttest_ind(conservation_df[(conservation_df.Conservation_type=='Kullback-Leibler')&(conservation_df.ele=='IDR')].dropna().conservation,
               conservation_df[(conservation_df.Conservation_type=='Kullback-Leibler')&(conservation_df.ele=='dSAM')].dropna().conservation))

print(stats.ttest_ind(conservation_df[(conservation_df.Conservation_type=='Kullback-Leibler')&(conservation_df.ele=='Structred_burid')].dropna().conservation,
               conservation_df[(conservation_df.Conservation_type=='Kullback-Leibler')&(conservation_df.ele=='dSAM')].dropna().conservation))

print(stats.ttest_ind(conservation_df[(conservation_df.Conservation_type=='lockless')&(conservation_df.ele=='IDR')].dropna().conservation,
               conservation_df[(conservation_df.Conservation_type=='lockless')&(conservation_df.ele=='dSAM')].dropna().conservation))

In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
ax.grid(color='#B4B5B4')
ax = sns.kdeplot(x='conservation',
            data=conservation_df[(conservation_df.Conservation_type=='Kullback-Leibler')&
                                 (conservation_df.ele!='Structred_exposed')&(conservation_df.ele=='IDR')],
            fill=1,lw=0,alpha=1,multiple="layer",
            hue='ele',palette=['#FF9B00','#2E4185','#C03830'],zorder=3)
ax.set_xlim([1,3.2])
ax.legend(['$mdSAM$','$Structured^{buried}$','$Disorder$'],fontsize=10,loc='lower center',frameon=False)
ax.set_xlabel('$Divergence$')
plt.locator_params(nbins=6)


In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
ax.grid(color='#B4B5B4')
ax = sns.kdeplot(x='conservation',
            data=conservation_df[(conservation_df.Conservation_type=='Kullback-Leibler')&
                                 (conservation_df.ele=='IDR')],
            fill=1,lw=2,alpha=.7,multiple="layer",
            hue='ele',palette=['#FF9B00'],zorder=3)
ax = sns.kdeplot(x='conservation',
            data=conservation_df[(conservation_df.Conservation_type=='Kullback-Leibler')&
                                 (conservation_df.ele.isin(['Structred_exposed','Structred_burid']))],
            fill=1,lw=2,alpha=.7,multiple="layer",
            palette=['#2E4185'],zorder=3)
ax = sns.kdeplot(x='conservation',
            data=conservation_df[(conservation_df.Conservation_type=='Kullback-Leibler')&
                                 (conservation_df.ele=='dSAM')],
            fill=1,lw=2,alpha=.7,multiple="layer",
            hue='ele',palette=['#C03830'],zorder=3,color='k')
ax.set_xlim([.55,3.3])
ax.legend(['$Disorder$','$Structured^{*}$','$mdSAM$'],fontsize=12,loc='upper left',frameon=False)
ax.set_xlabel('$Divergence$',fontsize=10)#(Kullback-Leibler)
ax.set_ylabel('$Density$',fontsize=10)
plt.locator_params(nbins=5)
#plt.title('$Kullback-Leibler$',fontsize=10)
plt.title('$Residue\;level$',y=1.0,fontsize=12)
plt.savefig('Conservation_new_version_v2.svg',bbox_inches='tight',transparent=True)


In [ ]:
conservation_df.loc[conservation_df[(conservation_df.Conservation_type=='Kullback-Leibler')&
                                 (conservation_df.ele.isin(['Structred_burid']))].index,'4_catag']='A'

In [ ]:
conservation_df

In [ ]:
#
conservation_df.loc[conservation_df[(conservation_df.Conservation_type=='shannon')&
                                 (conservation_df.ele=='IDR')].index,'4_catag']='$Disordered$'

conservation_df.loc[conservation_df[(conservation_df.Conservation_type=='shannon')&
                                 (conservation_df.ele.isin(['Structred_burid']))].index,'4_catag'] = '$Structred^{*}$'

conservation_df.loc[conservation_df[(conservation_df.Conservation_type=='shannon')&
                                 (conservation_df.ele.isin(['Structred_exposed']))].index,'4_catag'] = 'Structred_exposed'


conservation_df.loc[conservation_df[(conservation_df.Conservation_type=='shannon')&
                                 (conservation_df.ele=='dSAM')].index,'4_catag']='$mdSAM$'

In [ ]:
conservation_df['box']='A'

In [ ]:
fig, ax = plt.subplots(figsize=(3,3.5))

sns.boxplot(x='4_catag',y='conservation',data=conservation_df[(conservation_df['4_catag']!='Structred_exposed')&(conservation_df.Conservation_type=='Kullback-Leibler')],
            order=['$Disordered$','$mdSAM$','$Structred^{*}$'],
           palette=['#E69965','#C03830','#194D63'],notch=False,
           boxprops=dict(alpha=.95),showfliers = False,capprops=dict(linewidth=0))
#plt.legend(fontsize=9,frameon=False)
plt.xlabel('$Residue\;type$',fontsize=10)
plt.xticks(rotation=0,fontsize=10)
ax.set_ylabel('$Divergence$',fontsize=10)
plt.locator_params(axis='x', nbins=10)
ax.grid(color='#B4B5B4')
e = '%1.10e' % stats.ttest_ind(conservation_df[(conservation_df['4_catag']=='$Disordered$')&(conservation_df.Conservation_type=='Kullback-Leibler')].conservation.dropna(),
               conservation_df[(conservation_df['4_catag']=='$Structred^{*}$')&(conservation_df.Conservation_type=='Kullback-Leibler')].conservation.dropna())[-1]
lead = e[0]
power = e.split('-')[-1].split('0')[-1]
plt.text(1.2,3,'$%s⋅10^{‑%s}$'%(lead,power),rotation=0,fontsize=10,color='k')

e = '%1.10e' % stats.ttest_ind(conservation_df[(conservation_df['4_catag']=='$mdSAM$')&(conservation_df.Conservation_type=='Kullback-Leibler')].conservation.dropna(),
               conservation_df[(conservation_df['4_catag']=='$Structred^{*}$')&(conservation_df.Conservation_type=='Kullback-Leibler')].conservation.dropna())[-1]
lead = e[0]
power = e.split('-')[-1].split('0')[-1]
plt.text(1.2,2.75,'$%s⋅10^{‑%s}$'%(lead,power),rotation=0,fontsize=10,color='k')


e = '%1.10e' % stats.ttest_ind(conservation_df[(conservation_df['4_catag']=='$mdSAM$')&(conservation_df.Conservation_type=='Kullback-Leibler')].conservation.dropna(),
               conservation_df[(conservation_df['4_catag']=='$Disordered$')&(conservation_df.Conservation_type=='Kullback-Leibler')].conservation.dropna())[-1]
lead = e[0]
power = e.split('-')[-1].split('0')[-1]
plt.text(.8,2.5,'$%s⋅10^{‑%s}$'%(lead,power),rotation=0,fontsize=10,color='k')


plt.ylim([.5,3.5])
plt.savefig('Divergence.svg',bbox_inches='tight',transparent=True)

In [ ]:
conservation_df[conservation_df['4_catag']!='Structred_exposed']

In [ ]:
stats.ttest_ind(conservation_df[conservation_df['4_catag']=='$Disordered$'].conservation.dropna(),
               conservation_df[conservation_df['4_catag']=='$Structred^{*}$'].conservation.dropna())


In [ ]:
conservation_df[conservation_df['4_catag']=='$mdSAM$'].dropna()

In [ ]:
conservation_df.to_pickle('conservation_df.pkl')

In [ ]:
import re

''' a is the reference '''

b = 'MPQLYIYIRLLGAYLFIISRVQGQNLDSMLHGTGMKSDSDQKKSENGVTLAPEDTLPFLKCYCSGHCPDDAINNTCITNGHCFAIIEEDDQGETTLASGCMKYEGSDFQCKDSPKAQLRRTIECCRTNLCNQYLQPTLPPVVIGPFFDGSIRWLVLLISMAVCIIAMIIFSSCFCYKHYCKSISSRRRYNRDLEQDEAFIPVGESLKDLIDQSQSSGSGSGLPLLVQRTIAKQIQMVRQVGKGRYGEVWMGKWRGEKVAVKVFFTTEEASWFRETEIYQTVLMRHENILGFIAADIKGTGSWTQLYLITDYHENGSLYDFLKCATLDTRALLKLAYSAACGLCHLHTEIYGTQGKPAIAHRDLKSKNILIKKNGSCCIADLGLAVKFNSDTNEVDVPLNTRVGTKRYMAPEVLDESLNKNHFQPYIMADIYSFGLIIWEMARRCITGGIVEEYQLPYYNMVPSDPSYEDMREVVCVKRLRPIVSNRWNSDECLRAVLKLMSECWAHNPASRLTALRIKKTLAKMVESQDVKI'
a = 'MPQ----LYIYI---RLLGA------------------------YLFIISRVQGQNLDSMLHGTGMKSDSDQKKSEN----GVTLAPEDTLPFLKCYCSGHCPDDAINNTCIT--NGHCFAIIEEDDQGE-T----TLASGCMKY-----EGSDFQCK-DSP-KAQLRRTIECCR-TNLCNQYLQPTLPPVV-I------G--------PFFDGSIRWLVLLISMAVCIIAMIIFS-SCFCYKHYCKSISSRRRYNRDLEQDEAFI------------PVGESLKDLIDQSQSSGSGSGLPLLVQRTIAKQIQMVRQVGKGRYGEVWMGKWRGEKVAVKVFFTTEEASWFRETEIYQTVLMRHENILGFIAADIKG-TGSWTQLYLITDYHENGSLYDFLKC-ATLDTRALLKLAYSAACGLCHLHTEIYGTQ--GKPAIAHRDLKSKNILIKKNGSCCIADLGLAVKFNSDTNEVDVP-----LNTRVGTKRYMAPEVLDESLNKNHFQPYIMADIYSFGLIIWEMARRCITGG-----------------------------------------------IVEEYQLPYYNMVPSDPSYEDMREVVCVKRLRPIVSNRWNSDE---------------------------------------------------------------------------------CLRAVLKLMSECWAHNPASRLTALRIKKTLAKMVESQDVKI----------------------------'

''' b is 12-mer taken from the seqence of interest, in reality you'll be  doing this test for every possible 12-mer in the sequence'''
# b = 'TGT--CGAA-GG'

b = b.replace('-', '.')
r = re.compile(b);
m = r.match(a)

print(m)

In [ ]:
c = 0
u = []
for x in list(a):
    if x!='-':
        u.append('%d'%c)
        c =c +1 
    else:
        u.append('-')

In [ ]:
x = pd.read_pickle('/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/hTM_AF2_panther_orthologs/pkls/A0A022PP69.pkl')

In [ ]:
def msa3aln(acc):
    r = []
    c = 0
    u = []
    for x in list(a):
        if x!='-':
            u.append('%d'%c)
            c =c +1 
        else:
            u.append('-')
    for i in enumerate(u):
        if i[1] =='-':
            r.append({'acc':acc,'AA':'-','SAM':0,'pos':i[0]})
        else:
            r.append({'acc':acc,'AA':b[int(i[1])],'SAM':0,'pos':i[0]})
    return r 

In [ ]:
human_gene = pd.read_pickle('/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/webserver_sandbox/human_AF2/P48546.pkl')

In [ ]:
human_gene

In [ ]:
from Bio.Phylo.TreeConstruction import DistanceCalculator
from Bio.Phylo.TreeConstruction import DistanceTreeConstructor
from Bio import Phylo
def para2dist(acc,tree_cutoff=.7):#MSA 2 tree
    file = '/Volumes/aelazar//higher_order_assemblies/classifier/human_ORfs/AF2/hTM_AF2_panther_orthologs/MSAs/%s/%s_msa.fa'%(acc,acc)
    if os.path.isfile(file)==False:
        print('AA')
        return pd.DataFrame()
    #evo_dist = pd.DataFrame()
    
    fasta_sequences = SeqIO.parse(open(file),'fasta')
    # l = []
    # for fasta in fasta_sequences:
    #     name, sequence = fasta.id, str(fasta.seq)
    #     tax  = name.split('_')[1]
    #     l = l.append({'acc':name,'tax':tax,'AAseq':sequence})
    l = []
    for fasta in fasta_sequences:
        name, sequence = fasta.id, str(fasta.seq)
        tax = name.split('_')[1]
        l.append({'acc': name, 'tax': tax, 'AAseq': sequence})
    evo_dist = pd.DataFrame(l)
    calculator = DistanceCalculator('identity')
    msa = AlignIO.read(file, 'fasta')
    dm = calculator.get_distance(msa)
    # Construct the tree
    constructor = DistanceTreeConstructor()
    tree = constructor.nj(dm)  # You can use nj() or upgma()

    # Plot the tree
    Phylo.draw(tree)  # This uses matplotlib to render
    evo_dist['tree'] = dm[0]
    evo_dist = evo_dist.sort_values(by='tree')
    return evo_dist[evo_dist.tree<=tree_cutoff]

In [ ]:
import os
import pandas as pd
from Bio import SeqIO, AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

def para2dist_tree(acc, tree_cutoff=0.7):
    file = f'/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/hTM_AF2_panther_orthologs/MSAs/{acc}/{acc}_msa.fa'
    
    if not os.path.isfile(file):
        print('File not found.')
        return pd.DataFrame(), None

    # Extract sequence info
    fasta_sequences = list(SeqIO.parse(open(file), 'fasta'))
    l = []
    for fasta in fasta_sequences:
        name, sequence = fasta.id, str(fasta.seq)
        tax = name.split('_')[1]
        l.append({'acc': name, 'tax': tax, 'AAseq': sequence})
    evo_dist = pd.DataFrame(l)

    # Read MSA and compute distances
    msa = AlignIO.read(file, 'fasta')
    calculator = DistanceCalculator('identity')
    dm = calculator.get_distance(msa)

    # Build tree from distance matrix
    constructor = DistanceTreeConstructor()
    tree = constructor.nj(dm)  # or use upgma()

    # Compute distances from first sequence
    first_id = msa[0].id
    evo_dist['tree'] = [dm[first_id, record.id] for record in msa]
    evo_dist = evo_dist.sort_values(by='tree')
    
    # Filtered distances and tree returned
    return evo_dist[evo_dist.tree <= tree_cutoff], tree


In [ ]:
msa = para2dist('P08138')
msa

In [ ]:
_,tree = para2dist_tree('Q96RJ3')
Phylo.write(tree, 'tree-Q96RJ3.nwk', 'newick')

In [ ]:
_

In [ ]:
def msa3aln(acc,msa_seq):
    tax = acc.split('_')[-1]
    acc_ID = acc.split('_')[0]
    if tax =='HUMAN':
        pdb_pkl = pd.read_pickle('/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/webserver_sandbox/human_AF2/%s.pkl'%acc_ID)
    else:
        if os.path.isfile('/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/hTM_AF2_panther_orthologs/pkls/%s.pkl'%acc_ID)==False:
            return pd.DataFrame()
        pdb_pkl = pd.read_pickle('/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/hTM_AF2_panther_orthologs/pkls/%s.pkl'%acc_ID)
    no_gaps = msa_seq.replace("-","")
    r = []
    c = 0
    u = []
    for x in list(msa_seq):
        if x!='-':
            u.append('%d'%c)
            c =c +1 
        else:
            u.append('-')
    
    for i in enumerate(u):
        if i[1] =='-':
            r.append({'ID':tax,'AA':'-','SAM':0,'msa_pos':i[0],'tax_long':acc})
        else:
            if tax =='HUMAN':
                if pdb_pkl[(pdb_pkl.position==int(i[1]))&(pdb_pkl.AF2_pLDDT_max<70)&(pdb_pkl.AF2_SS_helix_count==0)&(pdb_pkl.AF2_accessibility=='***********')&(pdb_pkl.WT_Windows_above_thershold>0)].empty ==False:
                    r.append({'ID':tax,'AA':no_gaps[int(i[1])],'SAM':2,'msa_pos':i[0],'original_pos':int(i[1]),'tax_long':acc})
                else:
                    r.append({'ID':tax,'AA':no_gaps[int(i[1])],'SAM':1,'msa_pos':i[0],'original_pos':int(i[1]),'tax_long':acc})
            else:
                if pdb_pkl[(pdb_pkl.position==int(i[1]))&(pdb_pkl.WT_Windows_above_thershold>0)&(pdb_pkl.AF2_pLDDT_max<70)].empty ==False:
                    r.append({'ID':tax,'AA':no_gaps[int(i[1])],'SAM':2,'msa_pos':i[0],'original_pos':int(i[1]),'tax_long':acc})
                else:
                    r.append({'ID':tax,'AA':no_gaps[int(i[1])],'SAM':1,'msa_pos':i[0],'original_pos':int(i[1]),'tax_long':acc})
    return pd.DataFrame(r)

In [ ]:
o = []
for acc in tqdm(set(dSAMs.acc)):
    
    msa = para2dist(acc)#pd.Series(list(set.difference(set(TMs.acc),set(dSAMs.acc)))).sample(1).values[0])
    tt = pd.DataFrame()
    for i in msa.index:
        if msa3aln(msa.loc[i,'acc'],msa.loc[i,'AAseq']).empty ==True:
            continue
        else:
            tt = pd.concat([tt,msa3aln(msa.loc[i,'acc'],msa.loc[i,'AAseq'])],ignore_index=True)
    if tt.empty==False:
        for sam in dSAMs[dSAMs.acc==acc].index:

            min_  = tt[(tt.ID=='HUMAN')&(tt.SAM==2)&(tt.original_pos.isin(range(int(dSAMs.loc[sam,'start']),int(dSAMs.loc[sam,'end']))))].msa_pos.min()-20
            max_ = tt[(tt.ID=='HUMAN')&(tt.SAM==2)&(tt.original_pos.isin(range(int(dSAMs.loc[sam,'start']),int(dSAMs.loc[sam,'end']))))].msa_pos.min()+20
            if len(set(tt.ID)) -1 ==0:
                o.append({'SAM':dSAMs.loc[sam,'AAseq'],'pre':np.NaN,'number_of_tax':0})
            elif (np.isnan(min_)==False) and (np.isnan(max_)==False):
                pre = len(set(tt[(tt.msa_pos.isin(range(min_,max_)))&(tt.SAM==2)&(tt.ID!='HUMAN')].ID))/(len(set(tt.ID)) -1)*100
                o.append({'SAM':dSAMs.loc[sam,'AAseq'],'pre':pre,'number_of_tax':len(set(tt.ID)) -1})
                                

In [ ]:
# oo = []
for acc in tqdm(pd.Series(list(set(TMs.acc))).sample(1000)):
    pdb_pkl = pd.read_pickle('/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/webserver_sandbox/human_AF2/%s.pkl'%acc)
    msa = para2dist(acc)#pd.Series(list(set.difference(set(TMs.acc),set(dSAMs.acc)))).sample(1).values[0])
    tt = pd.DataFrame()
    for i in msa.index:
        if msa3aln(msa.loc[i,'acc'],msa.loc[i,'AAseq']).empty ==True:
            continue
        else:
            tt = pd.concat([tt,msa3aln(msa.loc[i,'acc'],msa.loc[i,'AAseq'])],ignore_index=True)
    if tt.empty==False:
        if len(pdb_pkl[(pdb_pkl.AF2_pLDDT_max<70)&(pdb_pkl.AF2_SS_helix_count==0)&(pdb_pkl.AF2_accessibility=='***********')])>10:
            min_ = int(pdb_pkl[(pdb_pkl.AF2_pLDDT_max<70)&(pdb_pkl.AF2_SS_helix_count==0)&(pdb_pkl.AF2_accessibility=='***********')].position.sample())-20
            max_ = min_+40
            
            if len(set(tt.ID)) -1 ==0:
                oo.append({'pre':np.NaN,'number_of_tax':0})
            if (np.isnan(min_)==False) and (np.isnan(max_)==False) and(min_>20):
                pre = len(set(tt[(tt.msa_pos.isin(range(min_,max_)))&(tt.SAM==2)&(tt.ID!='HUMAN')].ID))/(len(set(tt.ID)) -1)*100
                oo.append({'acc':acc,'pos':'%d-%d'%(min_,max_),'pre':pre,'number_of_tax':len(set(tt.ID)) -1})

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
sns.histplot(x='pre',data=oo[oo.number_of_tax>1],fill=True,alpha=1,color='#C03830',lw=0,kde=False,bins=10,ax=ax)

ax2 = plt.axes([0.3, 0.55, .3, .2], facecolor='w')
sns.stripplot(x='number_of_tax',data=oo[oo.number_of_tax>1],color='grey',alpha=.1,ax=ax2)

ax2.set_xscale('log')

#ax.set_xscale('log')
ax2.set_xlabel('$No.\;taxons\;in\;MSA$',fontsize=8)
ax.set_xlabel('$\%Orthologs\;with\;dSAM$')
ax.set_ylabel('$Count$')
#plt.savefig('across_orthologs.svg',bbox_inches='tight',transparent=True)


In [ ]:
oo = pd.DataFrame(o)

In [ ]:
g = sns.JointGrid(x='number_of_tax',y='pre',data=oo[oo.number_of_tax>1], space=0, ratio=17,
                 height=2)
g.plot_joint(sns.scatterplot,
             color="r", alpha=.2, legend=False,edgecolor= None)
g.plot_marginals(sns.rugplot, height=1, color="r", alpha=.2)
g.ax_joint.set_xscale('log')
g.ax_joint.grid(color='#B4B5B4')
plt.locator_params(axis='y', nbins=4)
g.ax_joint.set_ylabel('$\%Orthologs\;with\;dSAM$')
g.ax_joint.set_xlabel('$No.\;taxons\;in\;MSA$')
plt.savefig('across_orthologs_v2.svg',bbox_inches='tight',transparent=True)


In [ ]:
#oo.to_pickle('number_of_tax_vs_pre.pkl')
oo = pd.read_pickle('number_of_tax_vs_pre.pkl')

In [ ]:
fig, ax = plt.subplots(figsize=(30,3))

sns.pointplot(x='number_of_tax',y='pre',data=oo[oo.number_of_tax>1])
#sns.scatterplot(x='number_of_tax',y='pre',data=oo[oo.number_of_tax>1])

In [ ]:
m = []
for i in [1,30,61]:
    for j in oo[oo.number_of_tax.isin([i for i in range(i+1,i+31)])].pre:
        m.append({'range':'%d-%d'%(i+1,i+29),'m':j,'l':len(oo[oo.number_of_tax.isin([i for i in range(i+1,i+31)])])})

In [ ]:
import matplotlib

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
#ax.grid(color='#B4B5B4')
sns.set_palette("bright")
sns.boxplot(x='l',y='m',data=pd.DataFrame(m),hue='range',
            dodge=False,palette='binary_r',saturation=1,
           showfliers = False,whis=1,notch=True, showcaps=False)
ax.set_ylabel('$\%\,Orthologs\;with\;mdSAM$',fontsize=16)
ax.set_xlabel('$No.\,mdSAMs$',fontsize=16)
# sns.scatterplot(x='n',y='m',data=pd.DataFrame(m),hue='l',palette='viridis',edgecolor=None,size='l')

#norm = plt.Normalize()
# norm=matplotlib.colors.LogNorm(1,1160)
# sm = plt.cm.ScalarMappable(cmap="RdGy_r", norm=norm)#cividis"
# sm.set_array([])
plt.ylim([0,110])
ax.legend(title='$No.\;taxons\;in\;MSA$',fontsize=10)#,loc='upper left',frameon=False)

# Remove the legend and add a colorbar
#ax.get_legend().remove()
#ax.figure.colorbar(sm)
plt.gca().invert_xaxis()
plt.suptitle('$Motif\;level$',y=1.0,fontsize=24)
plt.savefig('across_orthologs_v2_v2.svg',bbox_inches='tight',transparent=True)


In [ ]:
plot_oo = oo[oo.number_of_tax>1].groupby('number_of_tax').mean()

In [ ]:
sns.kdeplot(x='number_of_tax',y='pre',data=oo[oo.number_of_tax>1],log_scale=(True,False))

In [ ]:
oo['with_sam'] = (oo[oo.number_of_tax>1]['pre'] * oo[oo.number_of_tax>1]['number_of_tax'])/100

In [ ]:
oo[pd.isna(oo.with_sam)]

In [ ]:
ee = oo[oo.number_of_tax>1].number_of_tax.value_counts().sort_values()

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
sns.scatterplot(x='number_of_tax',y='with_sam',data=oo[oo.number_of_tax>1],
                edgecolor=None,alpha=.25,s=10,color='#C03830',zorder=4)

# sns.scatterplot(x='number_of_tax',y='with_sam',data=oo[oo.pre.between(50,100)],
#                 edgecolor=None,alpha=.5,s=10,color='blue')
# sns.scatterplot(x='number_of_tax',y='with_sam',data=oo[oo.pre.between(25,50)],
#                 edgecolor=None,alpha=.5,s=10,color='green')
sns.lineplot(x=range(0,100),y=range(0,100),color='k')# plt.yscale('log')
sns.lineplot(x=range(0,100),y=[i/2 for i in range(0,100)],color='k')
sns.lineplot(x=range(0,100),y=[i/4 for i in range(0,100)],color='k')
sns.lineplot(x=range(0,100),y=[i/10 for i in range(0,100)],color='k')
#ax.grid(zorder=0)

plt.xlim([0,100])
plt.ylim([0,100])

#ax.locator_params(nbins=4)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
plt.xlabel('')
plt.ylabel('')
plt.savefig('msa_v2.svg',bbox_inches='tight',transparent=True)



In [ ]:
ee  = oo[oo.number_of_tax>1]
len(ee[ee.pre.between(50,100)])

In [ ]:
len(ee)

In [ ]:
for i in ee.index:
    plot_oo.loc[i,'number']=ee.loc[i]

In [ ]:
plot_oo.iloc[0:19].number.sum()/plot_oo.number.sum()

In [ ]:
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#3D5A80","#C03830"])

from matplotlib.colors import LogNorm, Normalize
fig, ax = plt.subplots(figsize=(3,3)) 

sns.scatterplot(x='number_of_tax',y='pre',
                data=plot_oo,hue='number',palette=cmap,edgecolor=None,s=70)
plt.legend([],frameon=False)
#norm = LogNorm(vmax=2,vmin=140)
norm = Normalize(vmax=136,vmin=2)#plt.Normalize(7, 12)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
ax.grid()
plt.xscale('log')
cbar = ax.figure.colorbar(sm,shrink=.50)
cbar.ax.tick_params(labelsize=10)
plt.xlabel('$No.\;sequences\;in\;MSA$',fontsize=10)
plt.ylabel('$\%\;Sequences\;with\;mdSAM$',fontsize=10)
cbar.set_label(label='$No.proteins$', size='x-small')
ax.fill_between(range(0,20), -10,120 
               ,color='gray', alpha=.2)
plt.ylim([0,105])
plt.text(3,90,'$80\%$')
plt.savefig('across_orthologs_v2_v3.svg',bbox_inches='tight',transparent=True)


In [ ]:
msa = para2dist('Q96RJ3')#Q8NC44

In [ ]:
#msa = para2dist('Q8NC44')#pd.Series(list(set.difference(set(TMs.acc),set(dSAMs.acc)))).sample(1).values[0])
tt = pd.DataFrame()
for i in msa.index:
    if msa3aln(msa.loc[i,'acc'],msa.loc[i,'AAseq']).empty ==True:
        continue
    else:
        tt = pd.concat([tt,msa3aln(msa.loc[i,'acc'],msa.loc[i,'AAseq'])],ignore_index=True)

In [ ]:
msa

In [ ]:
tt[tt.SAM==2].tax_long.drop_duplicates()

In [ ]:
def find_ranges(numbers):
    """
    Finds ranges of consecutive numbers in a list.

    Args:
      numbers: A list of numbers.

    Returns:
      A list of tuples, where each tuple represents a range of consecutive numbers.
    """
    if not numbers:
        return []

    ranges = []
    start = numbers[0]
    end = numbers[0]

    for i in range(1, len(numbers)):
        if numbers[i] == end + 1:
            end = numbers[i]
        else:
            ranges.append((start, end))
            start = numbers[i]
            end = numbers[i]
    ranges.append((start, end))
    return ranges

In [ ]:
find_ranges([int(i) for i in tt[(tt.SAM==2)&(tt.tax_long=='Q96RJ3_HUMAN')].original_pos])



In [ ]:
tt[(tt.SAM==2)&(tt.tax_long=='Q96RJ3_HUMAN')]

In [ ]:
print(''.join(tt[(tt.tax_long=='A0A139WIP4_TRICA')&(tt.original_pos.between(662-20,673+20))].AA.values))
print(''.join(tt[(tt.tax_long=='A0A139WIP4_TRICA')&(tt.original_pos.between(662,673))&(tt.SAM==2)].AA.values))


In [ ]:
import matplotlib
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w",'#b3b3b3','#b3b3b3','#9D282B'],N=3)
fig, ax = plt.subplots(figsize=(30, 5))

ax = sns.heatmap(tt.pivot(index='tax_long',columns='msa_pos',values='SAM').iloc[:,300:410].reindex(msa.acc),square=False,cmap=cmap,
                annot=tt.pivot(index='tax_long',columns='msa_pos',values='AA').iloc[:,300:410].reindex(msa.acc), fmt="")#,annot=pd.DataFrame(t).pivot("acc", "pos", "AA"), fmt="")
ax.set_xticklabels([])
#ax.set_yticklabels([])
ax.set_xticks([])
#ax.set_yticks([])
sns.despine(top=False,right=False)
plt.xlabel('$Position$')

plt.ylabel('$Taxons$')

In [ ]:
msa = para2dist('P21728')#pd.Series(list(set.difference(set(TMs.acc),set(dSAMs.acc)))).sample(1).values[0])
tt = pd.DataFrame()
for i in msa.index:
    if msa3aln(msa.loc[i,'acc'],msa.loc[i,'AAseq']).empty ==True:
        continue
    else:
        tt = pd.concat([tt,msa3aln(msa.loc[i,'acc'],msa.loc[i,'AAseq'])],ignore_index=True)

In [ ]:
hMP_AF_orthologdb#[hMP_AF_orthologdb.acc==name]['acc']

In [ ]:
min_  = tt[(tt.ID=='HUMAN')&(tt.SAM==2)&(tt.original_pos.isin(range(int(404.0),int(411.0))))].msa_pos.min()-20
max_ = tt[(tt.ID=='HUMAN')&(tt.SAM==2)&(tt.original_pos.isin(range(int(404.0),int(411.0))))].msa_pos.min()+20

In [ ]:
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w",'#b3b3b3','#b3b3b3','#9D282B'],N=3)
fig, ax = plt.subplots(figsize=(10, 10))
ax = sns.heatmap(tt[tt.msa_pos.isin(range(min_,max_))].pivot(index='ID',columns='msa_pos',values='SAM').reindex(msa.tax),square=True,cbar=True,cmap=cmap,lw=0,linecolor='k',
                annot=tt[tt.msa_pos.isin(range(min_,max_))].pivot("ID", "msa_pos", "AA").reindex(msa.tax), fmt="")

In [ ]:
TMs = pd.read_pickle('../files/hTMs_uniprot_v1.pkl')

In [ ]:
dSAMs[dSAMs.acc=='O14763']

In [ ]:
pdb_pkl = pd.read_pickle('/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/webserver_sandbox/human_AF2/%s.pkl'%'P29323')

In [ ]:
dSAMs[dSAMs.acc=='P48551']

In [ ]:
from joblib import Parallel, delayed

In [ ]:
import scipy.cluster.hierarchy as sch
import numpy as np
import radialtree as rt

np.random.seed(1)
labels=[chr(i)*10 for i in range(97, 97+numleaf)]
x = np.random.rand(numleaf)
D = np.zeros([numleaf,numleaf])
for i in range(numleaf):
    for j in range(numleaf):
        D[i,j] = abs(x[i] - x[j])

# Compute and plot the dendrogram.
Y = sch.linkage(D, method='single')
Z2 = sch.dendrogram(Y,labels=labels,no_plot=True)

# plot a circular dendrogram
rt.plot(Z2)